<a href="https://colab.research.google.com/github/boteny02/Research_Outcome/blob/main/ResNet101_Feature_Extraction_PCA_CNN_Classification_MRI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd


In [ ]:
image_data_labels = pd.read_excel('/content/drive/MyDrive/PhD_dataset/file_details_with_labels.xlsx')


1. **Data Setup and Preprocessing**: Load image metadata and labels from the Excel file, standardize the 'ExamDescription' column, verify image file paths to ensure only existing files are processed, and define a 'load_and_preprocess_image' function for consistent image preparation. This step ensures data readiness for feature extraction.
2. **ResNet101 Feature Extraction**: Load a pre-trained ResNet101 model (excluding its top classification layer) to serve as a feature extractor. Apply Global Average Pooling to generate a fixed-size feature vector for each valid MRI image, capturing high-level visual information.
3. **PCA Dimensionality Reduction**: Apply Principal Component Analysis (PCA) to the extracted ResNet101 features. This step aims to reduce the dimensionality of the feature space while retaining as much variance as possible, directly addressing the dimensionality curse. The optimal number of components will be determined based on explained variance.
4. **RoShuNet Integration (Clarification Needed)**: Integrate the 'RoShuNet algorithm' into the pipeline. Based on the context of 'dimensionality curse', this step is intended for further dimensionality reduction or specialized feature transformation. If 'RoShuNet' refers to a custom method or a less common technique, specific details will be required for accurate implementation.
5. **ReliefF Feature Selection**: Apply ReliefF feature selection to the dimensionally reduced features (from PCA and potentially RoShuNet). This technique will identify and select the most discriminative features that are highly relevant to the target labels, further refining the feature set.
6. **Data Splitting**: Split the preprocessed, dimensionally reduced, and feature-selected dataset into training and testing sets. This division is crucial for model development and unbiased evaluation of the classifier's performance.
7. **CNN Classification Model Training and Evaluation**: Build a simple dense neural network classifier using the prepared training features and labels. Compile and train the model, then evaluate its performance on the test set using metrics such as accuracy, precision, recall, F1-score, and AUC. Finally, visualize the confusion matrix to understand classification patterns.
8. **Final Task**: Provide a comprehensive summary of the implementation of PCA, its impact on dimensionality reduction, and the overall performance of the CNN classification model, including insights from the evaluation metrics and confusion matrix.

# Task
Load image metadata and labels from "/content/drive/MyDrive/PhD_dataset/file_details_with_labels.xlsx", preprocess the data by standardizing 'ExamDescription' and verifying image paths, then extract features using a pre-trained ResNet101 model. Apply PCA for dimensionality reduction and ReliefF for feature selection on these features. Split the dataset into training and testing sets, then train and evaluate a CNN classification model. Finally, provide a comprehensive summary of PCA's impact on dimensionality reduction and the overall performance of the CNN model using evaluation metrics and a confusion matrix.

## Data Setup and Preprocessing

### Subtask:
Load image metadata and labels from the Excel file, standardize the 'ExamDescription' column, verify image file paths to ensure only existing files are processed, and define a 'load_and_preprocess_image' function for consistent image preparation. This step ensures data readiness for feature extraction.


**Reasoning**:
The first part of the subtask involves standardizing the 'ExamDescription' column, constructing full image file paths, and filtering out non-existent image files. This is a sequence of data manipulation steps that can be efficiently performed in a single code block. I will import the `os` module for path verification.



In [ ]:
import os

# 1. Standardize the 'ExamDescription' column
image_data_labels['ExamDescription'] = image_data_labels['ExamDescription'].astype(str).str.lower().str.strip()

# Define the base directory for images
image_base_dir = '/content/drive/MyDrive/PhD_dataset/images/'

# 2. Construct the full image file paths
# Corrected: Removed 'Instance' as it caused a KeyError, using only available columns.
image_data_labels['image_path'] = image_data_labels.apply(lambda row:
    os.path.join(image_base_dir, f"P{row['Patient']}_E{row['Exam']}_S{row['Series']}.png"), axis=1
)

# 3. Verify the existence of each image file and filter the DataFrame
# Check if image file exists and create a boolean mask
image_data_labels['file_exists'] = image_data_labels['image_path'].apply(os.path.exists)

# Filter the DataFrame to include only rows where the image file exists
initial_rows = len(image_data_labels)
image_data_labels = image_data_labels[image_data_labels['file_exists']].copy()
final_rows = len(image_data_labels)

print(f"Initial number of image entries: {initial_rows}")
print(f"Number of existing image files found: {final_rows}")
print(f"Number of non-existent image files removed: {initial_rows - final_rows}")

# Drop the temporary 'file_exists' column
image_data_labels.drop(columns=['file_exists'], inplace=True)

print("ExamDescription standardized, image paths constructed, and non-existent image files removed.")
print("Updated image_data_labels head:")
print(image_data_labels.head())

**Reasoning**:
The previous code failed to find any image files, resulting in an empty DataFrame. This suggests an issue with the constructed file paths or the base directory. Before proceeding, I need to reload the original `image_data_labels` DataFrame from the Excel file. Then, I will inspect the `File_Path` and `Filename` columns, which were present in the initial DataFrame, as they are likely to contain the correct image file names or paths. I will use the `File_Path` column to directly create the `image_path`, assuming it holds the complete relative path to the image within the `image_base_dir`.



In [ ]:
import pandas as pd
import os

# Reload the original DataFrame as it became empty in the previous step
image_data_labels = pd.read_excel('/content/drive/MyDrive/PhD_dataset/file_details_with_labels.xlsx')

# 1. Standardize the 'ExamDescription' column
image_data_labels['ExamDescription'] = image_data_labels['ExamDescription'].astype(str).str.lower().str.strip()

# Define the base directory for images
image_base_dir = '/content/drive/MyDrive/PhD_dataset/images/'

# 2. Construct the full image file paths using the existing 'File_Path' column
# This assumes 'File_Path' contains the relative path from 'image_base_dir'
image_data_labels['image_path'] = image_data_labels['File_Path'].apply(lambda x: os.path.join(image_base_dir, x))

# 3. Verify the existence of each image file and filter the DataFrame
image_data_labels['file_exists'] = image_data_labels['image_path'].apply(os.path.exists)

# Filter the DataFrame to include only rows where the image file exists
initial_rows = len(image_data_labels)
image_data_labels = image_data_labels[image_data_labels['file_exists']].copy()
final_rows = len(image_data_labels)

print(f"Initial number of image entries: {initial_rows}")
print(f"Number of existing image files found: {final_rows}")
print(f"Number of non-existent image files removed: {initial_rows - final_rows}")

# Drop the temporary 'file_exists' column
image_data_labels.drop(columns=['file_exists'], inplace=True)

print("ExamDescription standardized, image paths constructed using 'File_Path', and non-existent image files removed.")
print("Updated image_data_labels head:")
print(image_data_labels.head())

**Reasoning**:
The previous steps successfully loaded the data, standardized 'ExamDescription', and created validated 'image_path' entries. The next part of the subtask requires defining a function to load and preprocess images for model input. This function will use TensorFlow for image loading, resizing, and normalization, and NumPy for array conversion.



In [ ]:
import tensorflow as tf
import numpy as np

# Define the target image size for ResNet101
IMG_HEIGHT = 224
IMG_WIDTH = 224

def load_and_preprocess_image(image_path):
    """
    Loads an image, resizes it, converts to a NumPy array,
    normalizes pixel values, and expands dimensions.
    """
    # Load the image
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    # Convert to NumPy array
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    # Normalize pixel values to [0, 1]
    img_array = img_array / 255.0
    # Expand dimensions to create a batch dimension (e.g., (1, H, W, C))
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

print("load_and_preprocess_image function defined.")

## ResNet101 Feature Extraction

### Subtask:
Load a pre-trained ResNet101 model (excluding its top classification layer) to serve as a feature extractor. Apply Global Average Pooling to generate a fixed-size feature vector for each valid MRI image, capturing high-level visual information.


**Reasoning**:
The subtask requires loading a pre-trained ResNet101 model, applying Global Average Pooling to its output, and then using this model to extract features from each image in the `image_data_labels` DataFrame. This involves defining the model and then iterating through the images to perform feature extraction, which can all be done in a single code block.



In [ ]:
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
import numpy as np
import time

# 1. Load a pre-trained ResNet101 model, excluding its top classification layer
baseline_model = ResNet101(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

# 2. Add Global Average Pooling layer
x = baseline_model.output
x = GlobalAveragePooling2D()(x)

# 3. Create a feature extractor model
feature_extractor = Model(inputs=baseline_model.input, outputs=x)

print("ResNet101 feature extractor model created.")

# 4. Initialize an empty list to store the extracted features
features_list = []

print("Starting feature extraction...")
start_time = time.time()

# 5. Iterate through each image_path and extract features
for index, row in image_data_labels.iterrows():
    image_path = row['image_path']

    # a. Preprocess the image
    preprocessed_image = load_and_preprocess_image(image_path)

    # b. Use the feature extractor model to get the feature vector
    feature_vector = feature_extractor.predict(preprocessed_image, verbose=0)

    # c. Append the extracted feature vector to the list
    features_list.append(feature_vector.flatten()) # Flatten to a 1D array

    if (index + 1) % 100 == 0:
        print(f"Processed {index + 1}/{len(image_data_labels)} images...")

# 6. Convert the list of extracted features into a NumPy array
image_features_resnet = np.array(features_list)

end_time = time.time()
elapsed_time = end_time - start_time

print(f"Feature extraction completed for {len(image_data_labels)} images in {elapsed_time:.2f} seconds.")
print(f"Shape of extracted features: {image_features_resnet.shape}")
print("First 5 rows of extracted features (first 5 dimensions):")
print(image_features_resnet[:5, :5])

## PCA Dimensionality Reduction

### Subtask:
Apply Principal Component Analysis (PCA) to the extracted ResNet101 features. This step aims to reduce the dimensionality of the feature space while retaining as much variance as possible, directly addressing the dimensionality curse. The optimal number of components will be determined based on explained variance.


**Reasoning**:
The subtask requires applying PCA to the extracted ResNet101 features, determining the optimal number of components based on explained variance, and then transforming the data. This involves importing PCA, fitting it to the features, calculating explained variance, and then applying the transformation, all of which can be done in a single code block.



In [ ]:
from sklearn.decomposition import PCA
import numpy as np

print("Starting PCA dimensionality reduction...")

# 1. Instantiate PCA object with n_components=None to compute all principal components
pca = PCA(n_components=None)

# 2. Fit the PCA model to the image_features_resnet array
pca.fit(image_features_resnet)

# 3. Calculate the cumulative explained variance ratio
cum_explained_variance = np.cumsum(pca.explained_variance_ratio_)

# 4. Determine the number of components needed to explain 95% of the variance
# We want to find the first index where cum_explained_variance exceeds 0.95 (95%)
optimal_components = np.where(cum_explained_variance >= 0.95)[0][0] + 1

print(f"Number of components to explain 95% variance: {optimal_components}")

# 5. Re-instantiate the PCA object with the optimal number of components
pca_final = PCA(n_components=optimal_components)

# 6. Apply the PCA transformation to the image_features_resnet array
image_features_pca = pca_final.fit_transform(image_features_resnet)

# 7. Print the shape of the image_features_pca array
print(f"Original feature shape: {image_features_resnet.shape}")
print(f"Reduced feature shape after PCA: {image_features_pca.shape}")

print("PCA dimensionality reduction completed.")

## ReliefF Feature Selection

### Subtask:
Apply ReliefF feature selection to the dimensionally reduced features (from PCA). This technique will identify and select the most discriminative features that are highly relevant to the target labels, further refining the feature set.


**Reasoning**:
The first step is to install the `sklearn-relief` library, which is required to perform ReliefF feature selection. This is a shell command that needs to be executed in a separate code block.



In [ ]:
pip install sklearn-relief

**Reasoning**:
The previous step successfully installed the `sklearn-relief` library. Now, I will complete the ReliefF feature selection by importing the `ReliefF` class, extracting the target labels, instantiating and fitting the `ReliefF` selector to the PCA-reduced features, and then transforming them. Finally, I will print the shape of the resulting feature set.



In [ ]:
from sklearn_relief import ReliefF
import numpy as np

print("Starting ReliefF feature selection...")

# 1. Extract the target labels from the image_data_labels DataFrame
y = image_data_labels['Label'].values

# 2. Instantiate the ReliefF selector
# The current PCA features are (5842, 13). Let's select 7 features as a reasonable reduction.
n_components_after_pca = image_features_pca.shape[1] # Total number of PCA features
n_features_for_relieff = min(7, n_components_after_pca) # Select 7 features or all if less than 7

# Corrected: Removed 'n_neighbors' and used 'n_features' parameter.
relief = ReliefF(n_features=n_features_for_relieff)

# 3. Fit the ReliefF selector to the image_features_pca and the target labels
# ReliefF expects X and y to be numpy arrays
relief.fit(image_features_pca, y)

# 4. Transform image_features_pca using the fitted ReliefF selector
# The 'transform' method will return the top N features as specified by 'n_features'
image_features_relieff = relief.transform(image_features_pca)

print(f"Original PCA feature shape: {image_features_pca.shape}")
print(f"Selected {n_features_for_relieff} features using ReliefF.")
print(f"Reduced feature shape after ReliefF: {image_features_relieff.shape}")

print("ReliefF feature selection completed.")

## Data Splitting

### Subtask:
Split the preprocessed, dimensionally reduced, and feature-selected dataset into training and testing sets. This division is crucial for model development and unbiased evaluation of the classifier's performance.


**Reasoning**:
The subtask requires splitting the dataset into training and testing sets. This involves importing the `train_test_split` function, extracting the target labels, and then applying the split to the features and labels while ensuring stratification and reproducibility. Finally, printing the shapes of the resulting sets will confirm the split was successful.



In [ ]:
from sklearn.model_selection import train_test_split

print("Starting data splitting...")

# 1. Extract the target labels from the image_data_labels DataFrame
y = image_data_labels['Label'].values

# 2. Split the image_features_relieff (features) and y (labels) into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    image_features_relieff, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Print the shapes of the resulting training and testing feature sets and label sets
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("Data splitting completed.")

## CNN Classification Model Training and Evaluation

### Subtask:
Build a simple dense neural network classifier using the prepared training features and labels. Compile and train the model, then evaluate its performance on the test set using metrics such as accuracy, precision, recall, F1-score, and AUC. Finally, visualize the confusion matrix to understand classification patterns.


**Reasoning**:
The subtask requires building, compiling, training, and evaluating a simple dense neural network classifier. This involves importing necessary TensorFlow and scikit-learn modules, defining the model architecture, compiling it with appropriate settings, training it on the `X_train` and `y_train` data, making predictions on `X_test`, calculating various evaluation metrics, and finally displaying a confusion matrix. All these actions can be performed sequentially within a single code block to complete the subtask.



In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

print("Starting CNN classification model training and evaluation...")

# 1. Get the number of features from X_train for input_shape
input_dim = X_train.shape[1]

# 2. Create a Sequential model
model = Sequential([
    Dense(128, activation='relu', input_shape=(input_dim,)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

# 3. Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()]
)

model.summary()

# 4. Train the model
history = model.fit(
    X_train, y_train,
    epochs=20, # Reduced epochs for quicker execution
    batch_size=32,
    validation_split=0.2, # Use a validation split to monitor performance
    verbose=1
)

# 5. Make predictions on the test set
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

# 6. Calculate and print evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

# 7. Generate and display a confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 6))
ConfusionMatrixDisplay(cm).plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

print("CNN classification model training and evaluation completed.")

**Reasoning**:
The previous code generated a `UserWarning` related to defining the input shape for the first `Dense` layer in a `Sequential` model. To address this, I will explicitly define an `Input` layer as the first layer of the `Sequential` model, which is the recommended practice in Keras.



In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt

print("Starting CNN classification model training and evaluation...")

# 1. Get the number of features from X_train for input_shape
input_dim = X_train.shape[1]

# 2. Create a Sequential model with an explicit Input layer
model = Sequential([
    Input(shape=(input_dim,)), # Explicit Input layer
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

# 3. Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()]
)

model.summary()

# 4. Train the model
history = model.fit(
    X_train, y_train,
    epochs=20, # Reduced epochs for quicker execution
    batch_size=32,
    validation_split=0.2, # Use a validation split to monitor performance
    verbose=1
)

# 5. Make predictions on the test set
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

# 6. Calculate and print evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

# 7. Generate and display a confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 6))
ConfusionMatrixDisplay(cm).plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

print("CNN classification model training and evaluation completed.")

## Final Task

### Subtask:
Provide a comprehensive summary of the implementation of PCA, its impact on dimensionality reduction, and the overall performance of the CNN classification model, including insights from the evaluation metrics and confusion matrix.


## Summary:

### Q&A
The implementation successfully used PCA for dimensionality reduction, reducing the feature space from 2048 dimensions to 13 dimensions while retaining 95% of the variance. ReliefF further refined this to 7 features. The CNN classification model achieved an accuracy of approximately 88.54% and an AUC of 0.9318, demonstrating good overall performance. However, a recall of 64.04% suggests that the model has room for improvement in identifying positive cases, as indicated by the confusion matrix.

### Data Analysis Key Findings
*   **Data Preparation**: Out of an initial set of image entries, 5842 valid image files were successfully located and verified for feature extraction, after standardizing `ExamDescription` and correcting image path construction.
*   **Feature Extraction**: A ResNet101 model extracted 2048-dimensional feature vectors for each of the 5842 images, resulting in a feature set shape of (5842, 2048).
*   **PCA Dimensionality Reduction**: Principal Component Analysis (PCA) successfully reduced the feature dimensionality from 2048 to 13 components, preserving 95% of the original variance.
*   **ReliefF Feature Selection**: ReliefF further refined the feature set, selecting 7 highly discriminative features from the 13 PCA components, leading to a final feature set shape of (5842, 7).
*   **Model Performance**: The CNN classification model achieved the following evaluation metrics on the test set:
    *   Accuracy: 0.8854
    *   Precision: 0.8657
    *   Recall: 0.6404
    *   F1-Score: 0.7362
    *   AUC: 0.9318
*   **Confusion Matrix Insights**: While the overall accuracy and AUC are high, the relatively lower recall (0.6404) indicates that the model struggles to correctly identify a significant portion of the actual positive cases (false negatives), even with good precision.

### Insights or Next Steps
*   The significant dimensionality reduction achieved by PCA (from 2048 to 13 features while retaining 95% variance) and subsequent ReliefF selection (to 7 features) was highly effective in preparing a compact and relevant feature set, potentially mitigating the curse of dimensionality and reducing computational cost.
*   To improve the model's ability to identify positive cases (recall), future steps could explore techniques to address class imbalance (if present), such as oversampling the minority class, using weighted loss functions, or experimenting with different model architectures or hyperparameter tuning.
